Preparação do Ambiente

In [ ]:
#Preparação do ambiente para rodar o Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/opt/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init('/opt/spark-3.3.0-bin-hadoop3')

from pyspark.sql import SparkSession

In [ ]:
# Sessão Spark
spark = SparkSession.builder \
    .appName("EcommerceDataLake") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/home/tavares/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

Criação da tabela principal

In [ ]:
spark.sql("""
CREATE TABLE hadoop_catalog.default.vendas_ecommerce (
    venda_id INT,
    produto_nome STRING,
    categoria STRING,
    quantidade INT,
    preco_unitario DOUBLE,
    data_venda DATE,
    cliente_id STRING,
    vendedor_id INT
)
USING iceberg
PARTITIONED BY (year(data_venda), categoria)
""").show()

In [ ]:
#Validação da tabela
spark.sql("""
    DESCRIBE FORMATTED hadoop_catalog.default.vendas_ecommerce
""").show(truncate=False)

Inserção de dados

In [ ]:
#Inserção de Dados Históricos 2023

spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (1, 'Notebook Dell', 'Eletrônicos', 2, 2500.00, DATE('2023-01-15'), 'CLI001', 101),
    (2, 'Mouse Logitech', 'Eletrônicos', 5, 80.00, DATE('2023-01-16'), 'CLI002', 102),
    (3, 'Mesa Escritório', 'Móveis', 1, 800.00, DATE('2023-02-10'), 'CLI003', 101),
    (4, 'Cadeira Gamer', 'Móveis', 2, 600.00, DATE('2023-02-15'), 'CLI001', 103),
    (5, 'Smartphone Samsung', 'Eletrônicos', 1, 1200.00, DATE('2023-03-20'), 'CLI004', 102)
""")

In [ ]:
#Inserção de dados 2024

spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (6, 'Tablet iPad', 'Eletrônicos', 1, 3000.00, DATE('2024-01-10'), 'CLI002', 101),
    (7, 'Sofá 3 Lugares', 'Móveis', 1, 1500.00, DATE('2024-01-20'), 'CLI005', 103),
    (8, 'Monitor 4K', 'Eletrônicos', 2, 800.00, DATE('2024-02-05'), 'CLI003', 102)
""")

Analise de Snapshots

In [ ]:
#Listando os snapshots criados ou seja ações realizadas na tabela
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce.snapshots").show()

In [ ]:
#Contando a quantidade de snapshots/ação na tabela
spark.sql("SELECT operation, COUNT(*) as num_operacoes FROM hadoop_catalog.default.vendas_ecommerce.snapshots GROUP BY operation;").show()

In [ ]:
#Visualizando historico da tabela
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce.history").show(truncate=False)

In [ ]:
#Acessando a primeira inserção de dados (snapshot)
snapshots = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at")
second_snapshot = snapshots.collect()[0][0]  # Segundo snapshot (índice 1)
print(f"ID do segundo snapshot: {second_snapshot}")

# Consultar dados como estavam no segundo snapshot
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF {second_snapshot}
    ORDER BY venda_id
""").show()

Time travel

In [ ]:
#Acessando a primeira inserção de dados (snapshot)
snapshots = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at")
second_snapshot = snapshots.collect()[0][0]  # Primeiro snapshot (índice o)
snapshot_atual = snapshots.collect()[3][0]  #snapshot atual (índice o)
print(f"ID do primeiro snapshot: {second_snapshot}")
print(f"ID do ultimo snapshot: {snapshot_atual}")

# Apenas os dados de 2023 usando o primeiro snapshot
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF {second_snapshot}
    ORDER BY venda_id
""").show()

# Apenas os dados de 2023 usando o primeiro snapshot
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF {snapshot_atual}
    ORDER BY venda_id
""").show()

Evolução de schema (add columns)

In [ ]:
# Adicionando novas colunas
spark.sql("""
    ALTER TABLE hadoop_catalog.default.vendas_ecommerce ADD COLUMNS desconto DOUBLE, canal_venda STRING
""")

In [ ]:
#Validando a dição das novas colunas
spark.sql("""
    DESCRIBE FORMATTED hadoop_catalog.default.vendas_ecommerce
""").show(truncate=False)

In [ ]:
#Inserção com Novo Schema
spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, DATE('2024-03-15'), 'CLI006', 101, 10.0, 'online'),
    (10, 'Mesa Centro', 'Móveis', 1, 400.00, DATE('2024-03-20'), 'CLI007', 102, 5.0, 'loja_fisica')
""").show(truncate=False)

In [ ]:
#Validando a inserção
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
""").show()

In [ ]:
#Mostre que dados antigos têm valores NULL nas novas colunas
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce where data_venda = '2023-01-15'
""").show()

Operações ACID e Merge

In [ ]:
#Atualizando o valor dos elestronicos
spark.sql(f"""
    UPDATE hadoop_catalog.default.vendas_ecommerce 
    SET preco_unitario = preco_unitario * 100 
    WHERE categoria = 'Eletrônicos'
""").show()

In [ ]:
#Validando a mudança de valores nos eletronicos
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
""").show()

Rollback

In [ ]:
#Visualiando snapshots
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce.history").show(truncate=False)

In [ ]:
# Consultar dados do snapshot anterior (antes da atualização problemática)
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    VERSION AS OF '2475487030487123182'
    ORDER BY venda_id
""").show()

In [ ]:
# Fazer rollback para o snapshot anterior
spark.sql(f"""
    CALL hadoop_catalog.system.rollback_to_snapshot(
        'hadoop_catalog.default.vendas_ecommerce', 
        2475487030487123182
    )
""")

In [ ]:
# Verificar dados após rollback
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce ORDER BY venda_id").show()

MERGE INTO

In [ ]:
#Criando uma view
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW vendas_updates AS
    SELECT 1 as venda_id, 'Notebook Dell UPDATED' as produto_nome, 'Eletrônicos' as categoria, 
           2 as quantidade, 2600.00 as preco_unitario, DATE('2023-01-15') as data_venda, 
           'CLI001' as cliente_id, 101 as vendedor_id, 0.0 as desconto, 'online' as canal_venda
    UNION ALL
    SELECT 11 as venda_id, 'Teclado Mecânico' as produto_nome, 'Eletrônicos' as categoria,
           4 as quantidade, 300.00 as preco_unitario, DATE('2024-04-01') as data_venda,
           'CLI008' as cliente_id, 103 as vendedor_id, 15.0 as desconto, 'online' as canal_venda
""")

In [ ]:
#Atualizando a tabela principal com o merge a partir da view
spark.sql("""
    MERGE INTO hadoop_catalog.default.vendas_ecommerce AS target
    USING vendas_updates AS source
    ON target.venda_id = source.venda_id
    WHEN MATCHED THEN
        UPDATE SET 
            target.venda_id = source.venda_id,
            target.produto_nome = source.produto_nome,
            target.categoria = source.categoria,
            target.quantidade = source.quantidade,
            target.preco_unitario = source.preco_unitario,
            target.data_venda = source.data_venda,
            target.cliente_id = source.cliente_id,
            target.vendedor_id = source.vendedor_id,
            target.desconto = source.desconto,
            target.canal_venda = source.canal_venda

    WHEN NOT MATCHED THEN 
        INSERT (venda_id, produto_nome, categoria, quantidade, preco_unitario, data_venda, cliente_id, vendedor_id, desconto, canal_venda)
        VALUES (source.venda_id, source.produto_nome, source.categoria, source.quantidade, source.preco_unitario, source.data_venda, source.cliente_id, source.vendedor_id, source.desconto, source.canal_venda);
""")

print("MERGE INTO executado com sucesso")

# Verificar resultados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce").show(truncate=False)

Análise, compactação e Otimização

In [ ]:
#Analisando manisfest (metadados do arquivo)
spark.sql("""
    SELECT 
        path,
        length,
        added_data_files_count
    FROM hadoop_catalog.default.vendas_ecommerce.manifests
""").show(truncate=True)

In [ ]:
#Analise de metadados da tabela/arquivos/tamanho em bytes desses arquivos
spark.sql("""
    SELECT 
        COUNT(*) as total_manifests,
        SUM(added_data_files_count) as total_arquivos_adicionados,
        SUM(existing_data_files_count) as total_arquivos_existentes,
        SUM(deleted_data_files_count) as total_arquivos_deletados,
        AVG(length) as tamanho_medio_manifest
    FROM hadoop_catalog.default.vendas_ecommerce.manifests
""").show()

In [ ]:
#Quantidade de registros por arquivo
df_files = spark.sql("""
    SELECT
        COUNT(*) AS registros,
        input_file_name() AS arquivo
    FROM hadoop_catalog.default.vendas_ecommerce
    GROUP BY input_file_name()
""")
df_files.show(truncate=True)

num_arquivos = df_files.count()
total_registros = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce").collect()[0][0]

print(f"- Número total de arquivos: {num_arquivos}")
print(f"- Total de registros: {total_registros}")

In [ ]:
# Informações detalhadas dos arquivos usando metadados Iceberg
print("=== INFORMAÇÕES DOS ARQUIVOS (METADADOS ICEBERG) ===")

spark.sql("""
    SELECT 
        file_path,
        file_format,
        record_count,
        file_size_in_bytes,
        ROUND(file_size_in_bytes / 1024.0, 2) as file_size_kb
    FROM hadoop_catalog.default.vendas_ecommerce.files
    ORDER BY file_path
""").show(truncate=False)

# Estatísticas agregadas
spark.sql("""
    SELECT 
        COUNT(*) as total_files,
        SUM(record_count) as total_records,
        AVG(record_count) as avg_records_per_file,
        SUM(file_size_in_bytes) as total_size_bytes,
        AVG(file_size_in_bytes) as avg_file_size_bytes
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").show(truncate=True)

In [ ]:
#Compactação de dados
# Configurar tamanho máximo de registros por arquivo
spark.conf.set("spark.sql.files.maxRecordsPerFile", 1000)

print("Executando compactação...")

# Compactação com procedimento 'rewrite_data_files'
result = spark.sql("""
    CALL hadoop_catalog.system.rewrite_data_files(
        table => 'default.vendas_ecommerce'
    )
""")

# Mostrar resultado da compactação
print("\nResultado da compactação:")
result.show()

In [ ]:
#Analise de performance
#Filtrando dados por partição
spark.sql("""SELECT * FROM hadoop_catalog.default.vendas_ecommerce WHERE year(data_venda) = 2024""").show(truncate=True)

In [ ]:
#Analise de performance
#Filtrando dados por partição e categoria
spark.sql("""SELECT * FROM hadoop_catalog.default.vendas_ecommerce WHERE year(data_venda) = 2024 and categoria = 'Eletrônicos'""").show(truncate=True)


In [ ]:
#Entendendo atraves de metadados(files) como podemos fazer melhores consultas a partir do armazenamento dos dados ou seja comportamento dos arquivos
spark.sql("""
    SELECT 
        partition,
        COUNT(*) AS arquivos_por_particao,
        SUM(record_count) AS registros_por_particao,
        SUM(file_size_in_bytes) AS tamanho_total_bytes
    FROM hadoop_catalog.default.vendas_ecommerce.files
    GROUP BY partition
    ORDER BY registros_por_particao DESC
""").show(truncate=False)

In [ ]:
#Verificando snapshot antes
spark.sql("""
    SELECT * 
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
""").show(truncate=True)

In [ ]:
#Limpeza de Snapshots (vaccum)
spark.sql("""
    CALL hadoop_catalog.system.expire_snapshots(
        table => 'default.vendas_ecommerce',
        retain_last => 3
    )
""")

In [ ]:
#Validando quantos arquivos foram removidos
spark.sql("""
    SELECT COUNT(*) AS total_snapshots
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
""").show()

Relatório de Análise

1. **Resumo Executivo**:
   - Quantos snapshots foram criados no total?
     R: 6 Snapshots.

   - Qual foi a redução de arquivos após compactação?
     R: Compactação não foi realizada pois foram gerados poucos arquivos por partição.

   - Quantas partições foram criadas?
     R: 4 partições

2. **Benefícios Observados**:
   - Liste 3 vantagens do Iceberg que você observou na prática
     R: Permite time travel e rollback sem recriar a tabela inteira.
        Visibilidade e rastreabilidade dos dados, permitindo analisar o que impacta o ETL.
        Só realiza atividades realmente necessárias, como compactação, evitando processamento desnecessário.

   - Compare com o que seria necessário usando Parquet tradicional
     R: Vejo o Parquet como apenas um bom formato de armazenamento. Qualquer operação rollback, merges, evolução de schema precisa de ferramentas externas, aumentando a complexidade. Já o Iceberg, utilizando o Parquet como formato, adiciona camada de gerenciamento, tornando essas operações bem mais simples.

3. **Casos de Uso Identificados**:
   - Descreva 2 cenários empresariais onde este pipeline seria útil
     R: Monitoramento de estoque e faturamento com atualizações frequentes de registros.
        Acompanhamento diário de vendas, promoções e comportamento de clientes.

   - Explique como o versionamento ajudaria em cada caso
     R: Possibilita recuperar versões anteriores após erros de inserção ou alterações de produtos/valores indevidas.
        Permite voltar no tempo para analisar dados históricos, exemplo um periodo especifico.
     

## 🏆 **ENTREGA FINAL**

### **Relatório de Análise**
Crie uma célula markdown final com:

1. **Resumo Executivo**:
   - Quantos snapshots foram criados no total?
   - Qual foi a redução de arquivos após compactação?
   - Quantas partições foram criadas?

2. **Benefícios Observados**:
   - Liste 3 vantagens do Iceberg que você observou na prática
   - Compare com o que seria necessário usando Parquet tradicional

3. **Casos de Uso Identificados**:
   - Descreva 2 cenários empresariais onde este pipeline seria útil
   - Explique como o versionamento ajudaria em cada caso


## 🏁 **Entrega**

Salve o notebook `exercicio_final.ipynb` com todas as tarefas completas e documentadas. O exercício deve demonstrar domínio prático dos conceitos fundamentais do Apache Iceberg aplicados em um cenário empresarial realista.